In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
NOTEBOOK_START = time.perf_counter()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
np.set_printoptions(suppress=True)


def log_stage(stage_name, start_time):
    elapsed = time.perf_counter() - start_time
    print(f"{stage_name}: {elapsed:.4f}s")
    return elapsed


print("✅ Libraries loaded")
print(f"✅ Global seed set: {SEED}")

## Integration with Scoring Engine

Replace the manual PD formula in `ScoringEngine.compute_probability_of_default()` with:

```python
# OLD (manual formula):
pd = 0.03 + (emi_ratio * 0.35) + (debt_ratio * 0.25) + ...

# NEW (ML-based):
applicant_features = {
    'monthly_revenue': monthly_revenue,
    'total_debt': total_debt,
    'monthly_emi': emi,
    'business_age_months': business_age_months,
    'gst_compliant': 1 if gst_compliant else 0,
    'has_disputes': 1 if has_disputes else 0
}
pd = predict_probability_of_default(applicant_features)['pd']
```

In [ ]:
start_time = time.perf_counter()

data_path = Path("ml/data/test_data.csv")
df = pd.read_csv(data_path)

rename_map = {
    "debt": "total_debt",
    "emi": "monthly_emi",
    "business_age": "business_age_months",
    "gst_compliance": "gst_compliant",
    "past_disputes": "disputes",
    "dispute": "disputes",
    "default_label": "default_label",
}
df = df.rename(columns=rename_map)

expected_columns = [
    "monthly_revenue",
    "total_debt",
    "monthly_emi",
    "business_age_months",
    "gst_compliant",
    "disputes",
]
missing_columns = [column for column in expected_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

for column in expected_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

if "default_label" in df.columns:
    df["default_label"] = pd.to_numeric(df["default_label"], errors="coerce")

numeric_columns = ["monthly_revenue", "total_debt", "monthly_emi", "business_age_months"]
categorical_columns = ["gst_compliant", "disputes"]

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

for column in categorical_columns:
    mode_values = df[column].mode(dropna=True)
    fill_value = mode_values.iloc[0] if not mode_values.empty else 0
    df[column] = df[column].fillna(fill_value)

if "default_label" in df.columns:
    df = df.dropna(subset=["default_label"]).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(df.head(3))
log_stage("Data loading and cleaning", start_time)

## Step 8: Integration and Output Format

This section demonstrates how to integrate the model with the scoring engine.

In [ ]:
start_time = time.perf_counter()

revenue = df["monthly_revenue"].to_numpy(dtype=np.float64)
total_debt = df["total_debt"].to_numpy(dtype=np.float64)
monthly_emi = df["monthly_emi"].to_numpy(dtype=np.float64)
business_age = df["business_age_months"].to_numpy(dtype=np.float64)
gst_compliant = df["gst_compliant"].to_numpy(dtype=np.float64)
disputes = df["disputes"].to_numpy(dtype=np.float64)

revenue_safe = np.clip(revenue, 1.0, None)
emi_ratio_raw = monthly_emi / revenue_safe
debt_ratio_raw = total_debt / (revenue_safe * 12.0)

emi_ratio = np.clip(emi_ratio_raw, 0.0, 1.5)
debt_ratio = np.clip(debt_ratio_raw, 0.0, 2.0)
estimated_expenses = 0.6 * revenue
profit = revenue - estimated_expenses
profit_margin = np.divide(profit, revenue_safe, out=np.zeros_like(profit), where=revenue_safe > 0)
free_cash_flow = revenue - monthly_emi - estimated_expenses
fcf_ratio = np.divide(free_cash_flow, revenue_safe, out=np.zeros_like(free_cash_flow), where=revenue_safe > 0)
log_revenue = np.log1p(revenue)
high_emi_flag = (emi_ratio > 0.4).astype(np.int8)
high_debt_flag = (debt_ratio > 0.6).astype(np.int8)
low_revenue_flag = (revenue < 80000).astype(np.int8)
compliance_score = gst_compliant
repayment_score = (
    (1.0 - emi_ratio) * 0.4
    + (1.0 - debt_ratio) * 0.3
    + compliance_score * 0.2
    - (disputes > 0).astype(np.float64) * 0.1
)
repayment_score = np.clip(repayment_score, 0.0, 1.0)

if "default_label" in df.columns:
    y = pd.to_numeric(df["default_label"], errors="coerce").fillna(0).astype(np.int8).to_numpy()
    print("Using provided default_label column.")
else:
    y = (
        (emi_ratio_raw > 0.5)
        | (debt_ratio_raw > 0.7)
        | ((revenue < 80000) & (debt_ratio_raw > 0.4))
    ).astype(np.int8)
    flip_mask = np.random.rand(len(y)) < 0.07
    y[flip_mask] = 1 - y[flip_mask]
    print("Generated synthetic target with noise.")


df["emi_ratio"] = emi_ratio
df["debt_ratio"] = debt_ratio
df["estimated_expenses"] = estimated_expenses
df["profit"] = profit
df["profit_margin"] = profit_margin
df["free_cash_flow"] = free_cash_flow
df["fcf_ratio"] = fcf_ratio
df["log_revenue"] = log_revenue
df["high_emi_flag"] = high_emi_flag
df["high_debt_flag"] = high_debt_flag
df["low_revenue_flag"] = low_revenue_flag
df["repayment_score"] = repayment_score

target_rate = float(np.mean(y))
print(f"Target default rate: {target_rate:.2%}")
print(f"PD target signal rate: {float(np.mean((emi_ratio_raw > 0.5) | (debt_ratio_raw > 0.7) | ((revenue < 80000) & (debt_ratio_raw > 0.4)))):.2%}")
log_stage("Feature engineering and target creation", start_time)

## Step 7: Generate Probability of Default Predictions

In [ ]:
start_time = time.perf_counter()

feature_columns = [
    "emi_ratio",
    "debt_ratio",
    "profit_margin",
    "fcf_ratio",
    "log_revenue",
    "business_age_months",
    "gst_compliant",
    "disputes",
    "high_emi_flag",
    "high_debt_flag",
    "low_revenue_flag",
    "repayment_score",
]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan).copy()
X = X.fillna(X.median(numeric_only=True)).fillna(0)
X["gst_compliant"] = X["gst_compliant"].astype(np.int8)
X["disputes"] = X["disputes"].astype(np.int8)
X["high_emi_flag"] = X["high_emi_flag"].astype(np.int8)
X["high_debt_flag"] = X["high_debt_flag"].astype(np.int8)
X["low_revenue_flag"] = X["low_revenue_flag"].astype(np.int8)
y = pd.Series(y, name="default").astype(np.int8)

print("Feature matrix shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())
log_stage("Feature selection and preprocessing", start_time)

In [ ]:
start_time = time.perf_counter()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

rf_params = {
    "n_estimators": 120,
    "max_depth": 5,
    "class_weight": "balanced",
    "random_state": SEED,
    "n_jobs": -1,
}

importance_model = RandomForestClassifier(**rf_params)
importance_model.fit(X_train, y_train)

calibrated_model = CalibratedClassifierCV(
    estimator=RandomForestClassifier(**rf_params),
    method="sigmoid",
    cv=3,
)
calibrated_model.fit(X_train, y_train)

log_stage("Train/test split and calibrated model training", start_time)
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## Step 6: Model Evaluation and Validation

In [ ]:
start_time = time.perf_counter()

y_train_proba = calibrated_model.predict_proba(X_train)[:, 1]
y_test_pred = calibrated_model.predict(X_test)
y_test_proba = calibrated_model.predict_proba(X_test)[:, 1]

raw_train_proba = importance_model.predict_proba(X_train)[:, 1]
raw_test_proba = importance_model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)
raw_train_auc = roc_auc_score(y_train, raw_train_proba)
raw_test_auc = roc_auc_score(y_test, raw_test_proba)
precision = precision_score(y_test, y_test_pred, zero_division=0)
recall = recall_score(y_test, y_test_pred, zero_division=0)
cm = confusion_matrix(y_test, y_test_pred)

print("=" * 70)
print("MODEL EVALUATION")
print("=" * 70)
print(f"Raw Train AUC     : {raw_train_auc:.4f}")
print(f"Raw Test AUC      : {raw_test_auc:.4f}")
print(f"Calibrated Train AUC: {train_auc:.4f}")
print(f"Calibrated Test AUC : {test_auc:.4f}")
print(f"Precision         : {precision:.4f}")
print(f"Recall            : {recall:.4f}")
print("Confusion matrix:")
print(cm)
print("=" * 70)

if test_auc < 0.72:
    print("⚠️ AUC below the synthetic target; dataset quality is the limiting factor.")

log_stage("Evaluation", start_time)

## Step 5: Train Random Forest Model

In [ ]:
def _extract_value(applicant_data, *keys, default=0):
    for key in keys:
        if key in applicant_data and applicant_data[key] is not None:
            return applicant_data[key]
    return default


def build_applicant_frame(applicant_data):
    monthly_revenue = float(_extract_value(applicant_data, "monthly_revenue", "revenue"))
    total_debt = float(_extract_value(applicant_data, "total_debt", "debt"))
    monthly_emi = float(_extract_value(applicant_data, "monthly_emi", "emi"))
    business_age_months = float(_extract_value(applicant_data, "business_age_months", "business_age"))
    gst_compliant = int(_extract_value(applicant_data, "gst_compliant", "gst_compliance"))
    disputes = int(_extract_value(applicant_data, "disputes", "past_disputes"))

    revenue_safe = max(monthly_revenue, 1.0)
    emi_ratio = float(np.clip(monthly_emi / revenue_safe, 0.0, 1.5))
    debt_ratio = float(np.clip(total_debt / (revenue_safe * 12.0), 0.0, 2.0))
    estimated_expenses = 0.6 * monthly_revenue
    profit = monthly_revenue - estimated_expenses
    profit_margin = float(np.divide(profit, revenue_safe))
    free_cash_flow = monthly_revenue - monthly_emi - estimated_expenses
    fcf_ratio = float(np.divide(free_cash_flow, revenue_safe))
    log_revenue = float(np.log1p(monthly_revenue))
    high_emi_flag = int(emi_ratio > 0.4)
    high_debt_flag = int(debt_ratio > 0.6)
    low_revenue_flag = int(monthly_revenue < 80000)
    compliance_score = float(gst_compliant)
    repayment_score = float(
        np.clip(
            (1.0 - emi_ratio) * 0.4
            + (1.0 - debt_ratio) * 0.3
            + compliance_score * 0.2
            - (disputes > 0) * 0.1,
            0.0,
            1.0,
        )
    )

    feature_frame = pd.DataFrame(
        [{
            "emi_ratio": emi_ratio,
            "debt_ratio": debt_ratio,
            "profit_margin": profit_margin,
            "fcf_ratio": fcf_ratio,
            "log_revenue": log_revenue,
            "business_age_months": business_age_months,
            "gst_compliant": gst_compliant,
            "disputes": disputes,
            "high_emi_flag": high_emi_flag,
            "high_debt_flag": high_debt_flag,
            "low_revenue_flag": low_revenue_flag,
            "repayment_score": repayment_score,
        }],
        columns=feature_columns,
    )

    context = {
        "monthly_revenue": monthly_revenue,
        "total_debt": total_debt,
        "monthly_emi": monthly_emi,
        "business_age_months": business_age_months,
        "gst_compliant": gst_compliant,
        "disputes": disputes,
        "emi_ratio": emi_ratio,
        "debt_ratio": debt_ratio,
        "profit_margin": profit_margin,
        "fcf_ratio": fcf_ratio,
        "log_revenue": log_revenue,
        "repayment_score": repayment_score,
    }
    return feature_frame, context


def compute_rule_score(context):
    score = 0
    score += 25 if context["emi_ratio"] < 0.20 else 15 if context["emi_ratio"] < 0.40 else 0
    score += 25 if context["debt_ratio"] < 0.25 else 15 if context["debt_ratio"] < 0.60 else 0
    score += 20 if context["monthly_revenue"] >= 150000 else 10 if context["monthly_revenue"] >= 80000 else 0
    score += 15 if context["gst_compliant"] == 1 else 0
    score += 10 if context["business_age_months"] >= 36 else 5 if context["business_age_months"] >= 12 else 0
    score += 5 if context["disputes"] == 0 else 0
    return int(np.clip(score, 0, 100))


def score_applicant(applicant_data):
    feature_frame, context = build_applicant_frame(applicant_data)
    pd_value = float(calibrated_model.predict_proba(feature_frame)[0, 1])
    pd_value = float(np.clip(np.round(pd_value, 4), 0.0, 1.0))
    rule_score = compute_rule_score(context)
    final_score = float((1.0 - pd_value) * 100.0 * 0.6 + rule_score * 0.4)

    if pd_value < 0.10:
        risk_category = "Low Risk"
    elif pd_value < 0.25:
        risk_category = "Medium Risk"
    else:
        risk_category = "High Risk"

    if pd_value < 0.08 and rule_score > 70:
        decision = "Approve"
    elif pd_value < 0.20:
        decision = "Review"
    else:
        decision = "Reject"

    positive_factors = []
    if context["emi_ratio"] < 0.20:
        positive_factors.append("Low EMI burden")
    if context["debt_ratio"] < 0.25:
        positive_factors.append("Low debt burden")
    if context["business_age_months"] >= 36:
        positive_factors.append("Established business age")
    if context["gst_compliant"] == 1:
        positive_factors.append("GST compliant")
    if context["repayment_score"] >= 0.70:
        positive_factors.append("Strong repayment score")

    negative_factors = []
    if context["monthly_revenue"] < 80000:
        negative_factors.append("Low revenue")
    if context["debt_ratio"] > 0.60:
        negative_factors.append("High debt ratio")
    if context["disputes"] > 0:
        negative_factors.append("Disputes present")
    if context["emi_ratio"] > 0.40:
        negative_factors.append("High EMI burden")

    return {
        "pd": pd_value,
        "final_score": round(final_score, 2),
        "risk_category": risk_category,
        "decision": decision,
        "key_factors": {
            "positive": positive_factors[:4],
            "negative": negative_factors[:4],
        },
    }


sample_applicant = {
    "monthly_revenue": 60000,
    "total_debt": 25176,
    "monthly_emi": 439,
    "business_age_months": 153,
    "gst_compliant": 1,
    "disputes": 0,
}

print(json.dumps(score_applicant(sample_applicant), indent=2))

## Step 4: Data Preprocessing and Normalization

In [ ]:
importance_df = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": importance_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

print("Sorted feature importance:")
print(importance_df.to_string(index=False))

brier = brier_score_loss(y_test, y_test_proba)
print(f"\nCalibration check - Brier score: {brier:.4f}")

## Step 3: Feature Engineering

Create derived features as per the specification:
- `emi_ratio`: EMI stress indicator
- `debt_ratio`: Leverage indicator (annualized)
- `log_revenue`: Non-linear revenue effect
- `age_bucket`: Business maturity stage

In [ ]:
start_time = time.perf_counter()

try:
    import shap

    shap_sample = X_test.iloc[[0]]
    shap_explainer = shap.TreeExplainer(importance_model)
    shap_values = shap_explainer.shap_values(shap_sample)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    sample_shap = pd.Series(shap_values[0], index=feature_columns).sort_values(key=np.abs, ascending=False)
    print("SHAP sample explanation:")
    print(sample_shap.head(5).to_string())
except Exception as exc:
    print(f"SHAP unavailable: {exc}")

log_stage("Optional SHAP explanation", start_time)

In [ ]:
start_time = time.perf_counter()

output_dir = Path("ml/models")
output_dir.mkdir(parents=True, exist_ok=True)
artifact_path = output_dir / "pd_calibrated_model.joblib"

from joblib import dump

dump(
    {
        "model": calibrated_model,
        "feature_columns": feature_columns,
        "rule_score_version": "v2",
        "random_seed": SEED,
    },
    artifact_path,
)

print(f"Saved calibrated model artifact to: {artifact_path}")
log_stage("Model export", start_time)

## Step 2: Load and Explore Dataset

In [ ]:
total_elapsed = time.perf_counter() - NOTEBOOK_START
print("=" * 70)
print("TRAINING COMPLETE - FULL PIPELINE UPGRADE")
print("=" * 70)
print(f"Total notebook time: {total_elapsed:.4f}s")
print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC : {test_auc:.4f}")
print(f"Default rate: {float(y.mean()):.2%}")
print(f"Feature count: {len(feature_columns)}")
print("=" * 70)

## Step 1: Import Required Libraries

# Credit Scoring Pipeline Upgrade

Production-style, deterministic PD modeling with calibrated probabilities, engineered risk features, explainability, and structured scoring output.